# `news` package — test notebook

Exercises the [src/news](../src/news) package end-to-end **without any API keys or network access** by plugging a synthetic in-memory source into the collector.

Sections:
1. Tagger — alias-dictionary asset matching
2. Dedup — canonical URL, title hash, 64-bit SimHash
3. End-to-end — synthetic source → collector → JSONL store → range query
4. On-disk layout inspection
5. (Optional) Live RSS smoke test

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import news
from news import (
    Article, AssetMention, Collector, JsonlStore,
    NewsConfig, RawArticle, SourceSpec,
)
from news.dedup import canonicalize_url, article_id, hashes_for, hamming64
from news.tagging import AliasTagger

print('news package loaded from', Path(news.__file__).parent)

## 1. Tagger

In [ ]:
import json

aliases = json.loads((REPO_ROOT / 'configs' / 'news_aliases.json').read_text())
tagger = AliasTagger(aliases, case_sensitive=('NEAR',))

cases = [
    ('Bitcoin surges past 100k as ETH lags',
     'BTC and bitcoin both rallied today, while ethereum traded flat.'),
    ('SEC sues Solana foundation',
     'The lawsuit targets SOL and the Solana ecosystem.'),
    ('Tesla earnings beat estimates',
     'No crypto mentions in the article body whatsoever.'),
    ('NEAR Protocol launches mainnet upgrade',
     'NEAR validators voted unanimously. The road ahead is near.'),
]
for title, body in cases:
    mentions = tagger.tag(title, body)
    print(f'{title!r:60} -> {[(m.symbol, m.score, m.mentions) for m in mentions]}')

Note how the `NEAR Protocol` case picks up `NEAR` from the title and uppercase mid-body, but ignores the lowercase english `near` thanks to `case_sensitive=('NEAR',)`.

## 2. Dedup

In [ ]:
u1 = 'https://www.CoinDesk.com/markets/2026/05/17/btc-surges/?utm_source=x&ref=y'
u2 = 'https://coindesk.com/markets/2026/05/17/btc-surges'
print('canon u1:', canonicalize_url(u1))
print('canon u2:', canonicalize_url(u2))
print('same id :', article_id(canonicalize_url(u1)) == article_id(canonicalize_url(u2)))

a = hashes_for('Bitcoin surges past 100k', 'BTC rallied to new highs today.')
b = hashes_for('Bitcoin surges past 100k', 'BTC rallied to new highs today, traders cheer.')
c = hashes_for('Ethereum staking yields drop', 'ETH validator rewards have decreased.')
print('near-dup hamming (a vs b):', hamming64(int(a.shingle_hash, 16), int(b.shingle_hash, 16)))
print('unrelated  hamming (a vs c):', hamming64(int(a.shingle_hash, 16), int(c.shingle_hash, 16)))

## 3. End-to-end pipeline with a synthetic source

We build a fake `NewsSource` that yields a hand-crafted set of `RawArticle`s, register it via the lazy factory by monkey-patching `build_source`, and run the full `Collector` over it. This proves the orchestration (tag → dedup → write → re-read) without external dependencies.

In [ ]:
import tempfile
from collections.abc import Iterator
from datetime import datetime, timedelta, timezone

from news.sources.base import NewsSource
from news import sources as _sources_pkg
from news.config import SourceSpec

NOW = datetime.now(timezone.utc).replace(microsecond=0)

FIXTURES = [
    dict(url='https://example.com/a/btc-surges',
         title='Bitcoin surges past 100k',
         body='BTC rallied to new highs today.',
         age_h=2),
    # syndicated near-duplicate of the previous one (different host, same body)
    dict(url='https://mirror.example.org/a/btc-surges?utm_source=tw',
         title='Bitcoin surges past 100k',
         body='BTC rallied to new highs today.',
         age_h=2),
    dict(url='https://example.com/b/eth-staking',
         title='Ethereum staking yields drop to 3%',
         body='ETH validator rewards have decreased.',
         age_h=5),
    dict(url='https://example.com/c/tesla-earnings',
         title='Tesla earnings beat estimates',
         body='No crypto mentions whatsoever.',
         age_h=3),  # should be skipped (no assets)
    dict(url='https://example.com/d/sol-lawsuit',
         title='SEC sues Solana foundation',
         body='The lawsuit targets SOL and the Solana ecosystem.',
         age_h=10),
]

class SyntheticSource(NewsSource):
    def fetch(self, since: datetime, until: datetime) -> Iterator[RawArticle]:
        for fx in FIXTURES:
            ts = NOW - timedelta(hours=fx['age_h'])
            if ts < since or ts > until:
                continue
            yield RawArticle(
                source=self.name,
                url=fx['url'],
                published_at=ts,
                title=fx['title'],
                body=fx['body'],
                language='en',
                raw={'fixture': True},
            )

# Patch the factory so 'kind = "synthetic"' resolves to our class.
_orig_build = _sources_pkg.build_source
def _patched_build(spec: SourceSpec, asset_universe):
    if spec.kind == 'synthetic':
        return SyntheticSource(name=spec.name, credibility=spec.credibility,
                               asset_universe=asset_universe)
    return _orig_build(spec, asset_universe)
_sources_pkg.build_source = _patched_build
# Collector imports `build_source` by attribute lookup, so also patch the module-level reference.
import news.collector as _coll_mod
_coll_mod.build_source = _patched_build

tmpdir = Path(tempfile.mkdtemp(prefix='news_test_'))
cfg = NewsConfig(
    data_dir=tmpdir,
    aliases=aliases,
    sources=[SourceSpec(name='synthetic', kind='synthetic', credibility=0.9)],
    case_sensitive_aliases=('NEAR',),
    keep_raw=True,
    near_dup_hamming=4,
)
print('tmp data dir:', tmpdir)

In [ ]:
coll = Collector(cfg)
stats = coll.collect(since=NOW - timedelta(days=1), until=NOW)
print('first run :', stats)

# Re-run: every article should be skipped as already-existing.
stats2 = coll.collect(since=NOW - timedelta(days=1), until=NOW)
print('rerun     :', stats2)

Expectations for the first run:
- `fetched=5`, `written=4` (Tesla article skipped — no crypto assets)
- `skipped_no_assets=1`
- `skipped_near_dup=1` (the mirror BTC article is flagged via SimHash; it still gets written but with `duplicate_of` pointing at the canonical copy)

Second run: `written=0`, `skipped_existing=4`.

In [ ]:
store = JsonlStore(cfg.data_dir)
rows = list(store.iter_articles(since=NOW - timedelta(days=1), until=NOW))
for a in rows:
    print(f'{a.published_at}  {a.source:10}  assets={[(m.symbol, m.score) for m in a.assets]}  dup_of={a.duplicate_of}  {a.title!r}')

In [ ]:
# Filter by asset
btc_rows = list(store.iter_articles(since=NOW - timedelta(days=1), until=NOW, assets=['BTC']))
print(f'BTC-tagged articles: {len(btc_rows)}')
for a in btc_rows:
    print(' -', a.title)

## 4. On-disk layout

In [ ]:
for p in sorted(cfg.data_dir.rglob('*')):
    if p.is_file():
        rel = p.relative_to(cfg.data_dir)
        print(f'{p.stat().st_size:>6}  {rel}')

In [ ]:
# Peek at one normalized article record
import json
articles_dir = cfg.data_dir / 'articles' / 'synthetic'
jl = next(iter(articles_dir.glob('*.jsonl')))
first = json.loads(jl.read_text().splitlines()[0])
print(json.dumps(first, indent=2, default=str))

## 5. (Optional) Live RSS smoke test

Requires `feedparser` and network access. Skips silently otherwise. Writes into the same temp dir.

In [ ]:
try:
    import feedparser  # noqa: F401
    have_feedparser = True
except ModuleNotFoundError:
    have_feedparser = False

if have_feedparser:
    rss_cfg = NewsConfig(
        data_dir=cfg.data_dir,
        aliases=aliases,
        sources=[SourceSpec(
            name='cointelegraph_rss', kind='rss', credibility=0.7,
            options={'feeds': ['https://cointelegraph.com/rss']},
        )],
    )
    # Restore the original factory for this run
    _coll_mod.build_source = _orig_build
    rss_stats = Collector(rss_cfg).collect(since=NOW - timedelta(days=2), until=NOW)
    print('rss stats:', rss_stats)
    _coll_mod.build_source = _patched_build  # restore patch for any further cells
else:
    print('feedparser not installed — skipping. `uv add feedparser` to enable.')

In [ ]:
# Clean up the temp dir when done
import shutil
# shutil.rmtree(tmpdir)
print('tmp dir kept at:', tmpdir, '\n(uncomment shutil.rmtree to delete)')